In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

from paperscope.models import Chunk
from paperscope.retrieval import (
    dense_retrieve,
    bm25_retrieve,
    hybrid_retrieve,
    reranked_retrieve,
    smart_retrieve
)


CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.json"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

all_chunks = [Chunk(**item) for item in chunk_data]

from qdrant_client import QdrantClient

QDRANT_PATH = PROJECT_ROOT / "storage" / "qdrant"
COLLECTION_NAME = "research_papers"

client = QdrantClient(path=str(QDRANT_PATH))

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    chunk.text.lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

embedding_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda"
)

In [ ]:
evaluation_questions = [
    {
        "question": "How does Self-RAG decide when retrieval is necessary?",
        "expected_paper_id": "self_rag_asai_2023"
    },
    {
        "question": "What is late interaction in ColBERT?",
        "expected_paper_id": "colbert_khattab_2020"
    },
    {
        "question": "How does HyDE perform zero-shot dense retrieval?",
        "expected_paper_id": "hyde_gao_2023"
    },
    {
        "question": "What problem does RAPTOR address?",
        "expected_paper_id": "raptor_sarthi_2024"
    },
    {
        "question": "How does CRAG respond to poor retrieval results?",
        "expected_paper_id": "crag_yan_2024"
    },
    {
        "question": "What does Lost in the Middle show about long-context models?",
        "expected_paper_id": "lost_in_middle_liu_2023"
    },
    {
        "question": "How does DPR retrieve passages for open-domain question answering?",
        "expected_paper_id": "dpr_karpukhin_2020"
    },
    {
        "question": "What are the two RAG formulations introduced by Lewis et al.?",
        "expected_paper_id": "rag_lewis_2020"
    },
    {
        "question": "What retrieval techniques are discussed in the RAG survey?",
        "expected_paper_id": "rag_survey_gao_2023"
    },
    {
        "question": "What does the Qwen3 Embedding paper propose for embedding and reranking?",
        "expected_paper_id": "qwen3_embedding_2025"
    }
]

evaluation_questions = [
    # --------------------------------------------------
    # 1. Single-paper factual questions
    # --------------------------------------------------

    {
        "question": "What are the two RAG formulations introduced by Lewis et al.?",
        "expected_paper_id": "rag_lewis_2020",
        "category": "factual"
    },
    {
        "question": "What type of non-parametric memory does the original RAG model use?",
        "expected_paper_id": "rag_lewis_2020",
        "category": "factual"
    },
    {
        "question": "How does DPR represent questions and passages for retrieval?",
        "expected_paper_id": "dpr_karpukhin_2020",
        "category": "factual"
    },
    {
        "question": "Which datasets are used to evaluate DPR on open-domain question answering?",
        "expected_paper_id": "dpr_karpukhin_2020",
        "category": "factual"
    },
    {
        "question": "What does MaxSim do in ColBERT?",
        "expected_paper_id": "colbert_khattab_2020",
        "category": "factual"
    },
    {
        "question": "What is the main idea behind late interaction in ColBERT?",
        "expected_paper_id": "colbert_khattab_2020",
        "category": "factual"
    },
    {
        "question": "What is a hypothetical document in HyDE?",
        "expected_paper_id": "hyde_gao_2023",
        "category": "factual"
    },
    {
        "question": "How does Self-RAG decide when retrieval is necessary?",
        "expected_paper_id": "self_rag_asai_2023",
        "category": "factual"
    },
    {
        "question": "What are reflection tokens in Self-RAG?",
        "expected_paper_id": "self_rag_asai_2023",
        "category": "factual"
    },
    {
        "question": "What does the Lost in the Middle paper observe about information placed in the middle of long contexts?",
        "expected_paper_id": "lost_in_middle_liu_2023",
        "category": "factual"
    },

    # --------------------------------------------------
    # 2. Methodology / mechanism questions
    # --------------------------------------------------

    {
        "question": "How does HyDE perform zero-shot dense retrieval without relevance labels?",
        "expected_paper_id": "hyde_gao_2023",
        "category": "methodology"
    },
    {
        "question": "How does RAPTOR organize documents for retrieval?",
        "expected_paper_id": "raptor_sarthi_2024",
        "category": "methodology"
    },
    {
        "question": "How are recursive summaries created in RAPTOR?",
        "expected_paper_id": "raptor_sarthi_2024",
        "category": "methodology"
    },
    {
        "question": "How does CRAG evaluate whether retrieved documents are useful?",
        "expected_paper_id": "crag_yan_2024",
        "category": "methodology"
    },
    {
        "question": "What corrective actions does CRAG take when retrieval quality is poor?",
        "expected_paper_id": "crag_yan_2024",
        "category": "methodology"
    },
    {
        "question": "How does Self-RAG use retrieval and critique during inference?",
        "expected_paper_id": "self_rag_asai_2023",
        "category": "methodology"
    },
    {
        "question": "How does the original RAG architecture combine the retriever and generator?",
        "expected_paper_id": "rag_lewis_2020",
        "category": "methodology"
    },
    {
        "question": "How is similarity computed between query and document representations in ColBERT?",
        "expected_paper_id": "colbert_khattab_2020",
        "category": "methodology"
    },

    # --------------------------------------------------
    # 3. Paper-specific / metadata-aware questions
    # --------------------------------------------------

    {
        "question": "What retrieval techniques are discussed in the RAG survey?",
        "expected_paper_id": "rag_survey_gao_2023",
        "category": "paper_specific"
    },
    {
        "question": "What does the RAG survey describe as Naive RAG?",
        "expected_paper_id": "rag_survey_gao_2023",
        "category": "paper_specific"
    },
    {
        "question": "How does the RAG survey describe iterative retrieval?",
        "expected_paper_id": "rag_survey_gao_2023",
        "category": "paper_specific"
    },
    {
        "question": "What does the RAG survey say about modular RAG?",
        "expected_paper_id": "rag_survey_gao_2023",
        "category": "paper_specific"
    },
    {
        "question": "What limitations of long-context language models are discussed in Lost in the Middle?",
        "expected_paper_id": "lost_in_middle_liu_2023",
        "category": "paper_specific"
    },
    {
        "question": "What retrieval failure problem motivates Corrective RAG?",
        "expected_paper_id": "crag_yan_2024",
        "category": "paper_specific"
    },

    # --------------------------------------------------
    # 4. Exact-term / lexical questions
    # These are useful for testing whether BM25 helps.
    # --------------------------------------------------

    {
        "question": "What role does MaxSim play in ColBERT's late interaction mechanism?",
        "expected_paper_id": "colbert_khattab_2020",
        "category": "exact_term"
    },
    {
        "question": "How is the Retrieve reflection token used in Self-RAG?",
        "expected_paper_id": "self_rag_asai_2023",
        "category": "exact_term"
    },
    {
        "question": "What is retrieval collapse in the original RAG paper?",
        "expected_paper_id": "rag_lewis_2020",
        "category": "exact_term"
    },
    {
        "question": "How does RAPTOR use clustering when constructing its retrieval tree?",
        "expected_paper_id": "raptor_sarthi_2024",
        "category": "exact_term"
    },

    # --------------------------------------------------
    # 5. Embedding / reranking paper
    # --------------------------------------------------

    {
        "question": "What model family is introduced for text embedding and reranking in the Qwen3 Embedding paper?",
        "expected_paper_id": "qwen3_embedding_2025",
        "category": "embedding"
    },
    {
        "question": "How does the Qwen3 Embedding paper approach reranking?",
        "expected_paper_id": "qwen3_embedding_2025",
        "category": "embedding"
    },
]

In [ ]:
def evaluate_dense_recall_at_k(
    evaluation_questions,
    embedding_model,
    client,
    collection_name,
    k=5
):
    hits = 0
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_paper_id = item["expected_paper_id"]

        results = dense_retrieve(
            query=query,
            embedding_model=embedding_model,
            client=client,
            collection_name=collection_name,
            top_k=k
        )

        retrieved_papers = [
            point.payload["paper_id"]
            for point in results
        ]

        hit = expected_paper_id in retrieved_papers

        if hit:
            hits += 1

        rows.append({
            "question": query,
            "expected_paper": expected_paper_id,
            "retrieved_papers": retrieved_papers,
            "hit": hit
        })

    recall_at_k = hits / len(evaluation_questions)

    return recall_at_k, rows

In [ ]:
dense_recall_5, dense_rows = evaluate_dense_recall_at_k(
    evaluation_questions=evaluation_questions,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    k=5
)

print("Dense Recall@5:", dense_recall_5)

In [ ]:
for row in dense_rows:
    if not row["hit"]:
        print("QUESTION:", row["question"])
        print("EXPECTED:", row["expected_paper"])
        print("RETRIEVED:", row["retrieved_papers"])
        print("-" * 80)

In [ ]:
def evaluate_dense_mrr(
    evaluation_questions,
    embedding_model,
    client,
    collection_name,
    k=5
):
    reciprocal_ranks = []
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_paper_id = item["expected_paper_id"]

        results = dense_retrieve(
            query=query,
            embedding_model=embedding_model,
            client=client,
            collection_name=collection_name,
            top_k=k
        )

        retrieved_papers = [
            point.payload["paper_id"]
            for point in results
        ]

        reciprocal_rank = 0.0

        for rank, paper_id in enumerate(retrieved_papers, start=1):
            if paper_id == expected_paper_id:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "expected_paper": expected_paper_id,
            "retrieved_papers": retrieved_papers,
            "reciprocal_rank": reciprocal_rank
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, rows

In [ ]:
dense_mrr, dense_mrr_rows = evaluate_dense_mrr(
    evaluation_questions=evaluation_questions,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    k=5
)

print("Dense MRR:", dense_mrr)

In [ ]:
def evaluate_bm25_mrr(
    evaluation_questions,
    bm25,
    chunks,
    k=5
):
    reciprocal_ranks = []
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_paper_id = item["expected_paper_id"]

        results = bm25_retrieve(
            query=query,
            bm25=bm25,
            chunks=chunks,
            top_k=k
        )

        retrieved_papers = [
            result["chunk"].paper_id
            for result in results
        ]

        reciprocal_rank = 0.0

        for rank, paper_id in enumerate(retrieved_papers, start=1):
            if paper_id == expected_paper_id:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "expected_paper": expected_paper_id,
            "retrieved_papers": retrieved_papers,
            "reciprocal_rank": reciprocal_rank
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, rows

In [ ]:
bm25_mrr, bm25_mrr_rows = evaluate_bm25_mrr(
    evaluation_questions=evaluation_questions,
    bm25=bm25,
    chunks=all_chunks,
    k=5
)

print("BM25 MRR:", bm25_mrr)

In [ ]:
def evaluate_hybrid_mrr(
    evaluation_questions,
    embedding_model,
    client,
    collection_name,
    bm25,
    chunks,
    k=5
):
    reciprocal_ranks = []
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_paper_id = item["expected_paper_id"]

        results = hybrid_retrieve(
            query=query,
            embedding_model=embedding_model,
            client=client,
            collection_name=collection_name,
            bm25=bm25,
            chunks=chunks,
            candidate_k=10,
            top_k=k
        )

        retrieved_papers = [
            result["chunk"]["paper_id"]
            for result in results
        ]

        reciprocal_rank = 0.0

        for rank, paper_id in enumerate(retrieved_papers, start=1):
            if paper_id == expected_paper_id:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "expected_paper": expected_paper_id,
            "retrieved_papers": retrieved_papers,
            "reciprocal_rank": reciprocal_rank
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, rows

In [ ]:
hybrid_mrr, hybrid_mrr_rows = evaluate_hybrid_mrr(
    evaluation_questions=evaluation_questions,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    k=5
)

print("Hybrid MRR:", hybrid_mrr)

In [ ]:
def evaluate_reranked_mrr(
    evaluation_questions,
    embedding_model,
    client,
    collection_name,
    bm25,
    chunks,
    reranker,
    k=5
):
    reciprocal_ranks = []
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_paper_id = item["expected_paper_id"]

        results = reranked_retrieve(
            query=query,
            embedding_model=embedding_model,
            client=client,
            collection_name=collection_name,
            bm25=bm25,
            chunks=chunks,
            reranker=reranker,
            candidate_k=15,
            top_k=k
        )

        retrieved_papers = [
            result["chunk"]["paper_id"]
            for result in results
        ]

        reciprocal_rank = 0.0

        for rank, paper_id in enumerate(retrieved_papers, start=1):
            if paper_id == expected_paper_id:
                reciprocal_rank = 1 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "expected_paper": expected_paper_id,
            "retrieved_papers": retrieved_papers,
            "reciprocal_rank": reciprocal_rank
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, rows

In [ ]:
reranked_mrr, reranked_rows = evaluate_reranked_mrr(
    evaluation_questions=evaluation_questions,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    k=5
)

print("Reranked MRR:", reranked_mrr)

In [ ]:
for hybrid_row, rerank_row in zip(hybrid_mrr_rows, reranked_rows):
    if rerank_row["reciprocal_rank"] < hybrid_row["reciprocal_rank"]:
        print("QUESTION:", hybrid_row["question"])
        print("EXPECTED:", hybrid_row["expected_paper"])
        print("HYBRID:", hybrid_row["retrieved_papers"])
        print("RERANKED:", rerank_row["retrieved_papers"])
        print(
            "RR:",
            hybrid_row["reciprocal_rank"],
            "->",
            rerank_row["reciprocal_rank"]
        )
        print("-" * 100)

In [ ]:
query = "What retrieval techniques are discussed in the RAG survey?"

candidates = hybrid_retrieve(
    query=query,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    candidate_k=15,
    top_k=15
)

pairs = [
    (query, result["chunk"]["text"])
    for result in candidates
]

scores = reranker.predict(pairs)

comparison = []

for result, score in zip(candidates, scores):
    comparison.append({
        "paper": result["chunk"]["paper_id"],
        "section": result["chunk"]["section_heading"],
        "rrf_score": result["rrf_score"],
        "rerank_score": float(score)
    })

comparison.sort(
    key=lambda x: x["rerank_score"],
    reverse=True
)

for item in comparison:
    print(
        round(item["rerank_score"], 3),
        "|",
        item["paper"],
        "|",
        item["section"],
        "| RRF:",
        round(item["rrf_score"], 5)
    )

the reranker ignores the explicit “in the RAG survey” constraint and instead prefers chunks that are semantically dense about retrieval techniques.

In [ ]:
import importlib
import paperscope.retrieval

importlib.reload(paperscope.retrieval)
query = "What retrieval techniques are discussed in the RAG survey?"

from paperscope.retrieval import (
    dense_retrieve,
    bm25_retrieve,
    hybrid_retrieve,
    reranked_retrieve,
)
results = dense_retrieve(
    query=query,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    top_k=5,
    paper_id="rag_survey_gao_2023"
)

In [ ]:
for rank, point in enumerate(results, start=1):
    print(
        rank,
        round(point.score, 4),
        "|",
        point.payload["paper_id"],
        "|",
        point.payload["section_heading"]
    )

In [ ]:
import importlib
import paperscope.retrieval

importlib.reload(paperscope.retrieval)

from paperscope.retrieval import reranked_retrieve

In [ ]:
results = reranked_retrieve(
    query="What retrieval techniques are discussed in the RAG survey?",
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    paper_id="rag_survey_gao_2023",
    candidate_k=30,
    top_k=5
)

In [ ]:
for item in results:
    print(
        round(item["rerank_score"], 3),
        "|",
        item['chunk']["paper_id"],
        "|",
        item['chunk']["section_heading"],
        "| RRF:",
        round(item["rrf_score"], 5)
    )

In [ ]:
from collections import defaultdict


def category_mrr(rows, evaluation_questions):
    category_lookup = {
        item["question"]: item["category"]
        for item in evaluation_questions
    }

    scores = defaultdict(list)

    for row in rows:
        category = category_lookup[row["question"]]
        scores[category].append(row["reciprocal_rank"])

    return {
        category: sum(values) / len(values)
        for category, values in scores.items()
    }

In [ ]:
dense_by_category = category_mrr(
    dense_mrr_rows,
    evaluation_questions
)

bm25_by_category = category_mrr(
    bm25_mrr_rows,
    evaluation_questions
)

hybrid_by_category = category_mrr(
    hybrid_mrr_rows,
    evaluation_questions
)

reranked_by_category = category_mrr(
    reranked_rows,
    evaluation_questions
)

In [ ]:
categories = sorted(dense_by_category.keys())

for category in categories:
    print(category)
    print("  Dense:   ", round(dense_by_category[category], 3))
    print("  BM25:    ", round(bm25_by_category[category], 3))
    print("  Hybrid:  ", round(hybrid_by_category[category], 3))
    print("  Reranked:", round(reranked_by_category[category], 3))
    print()

In [ ]:
import json

evaluation_results = {
    "overall_mrr": {
        "dense": dense_mrr,
        "bm25": bm25_mrr,
        "hybrid": hybrid_mrr,
        "reranked": reranked_mrr,
    },
    "category_mrr": {
        "dense": dense_by_category,
        "bm25": bm25_by_category,
        "hybrid": hybrid_by_category,
        "reranked": reranked_by_category,
    }
}

RESULTS_PATH = PROJECT_ROOT / "data" / "evaluation" / "retrieval_metrics.json"

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_results,
        f,
        indent=2
    )

print("Saved to:", RESULTS_PATH)

In [ ]:
multi_paper_questions = [
    {
        "question": "Compare how Self-RAG and CRAG handle unreliable retrieval.",
        "expected_paper_ids": [
            "self_rag_asai_2023",
            "crag_yan_2024"
        ]
    },
    {
        "question": "How do RAPTOR and the original RAG paper differ in their retrieval approach?",
        "expected_paper_ids": [
            "raptor_sarthi_2024",
            "rag_lewis_2020"
        ]
    },
    {
        "question": "Compare DPR and ColBERT as neural retrieval methods.",
        "expected_paper_ids": [
            "dpr_karpukhin_2020",
            "colbert_khattab_2020"
        ]
    },
    {
        "question": "How do HyDE and DPR approach dense retrieval differently?",
        "expected_paper_ids": [
            "hyde_gao_2023",
            "dpr_karpukhin_2020"
        ]
    },
    {
        "question": "Compare Self-RAG and RAPTOR as approaches for improving retrieval-augmented generation.",
        "expected_paper_ids": [
            "self_rag_asai_2023",
            "raptor_sarthi_2024"
        ]
    },
    {
        "question": "How do CRAG and Self-RAG decide whether retrieved information should be trusted or used?",
        "expected_paper_ids": [
            "crag_yan_2024",
            "self_rag_asai_2023"
        ]
    },
    {
        "question": "What do Lost in the Middle and RAPTOR suggest about handling large amounts of context?",
        "expected_paper_ids": [
            "lost_in_middle_liu_2023",
            "raptor_sarthi_2024"
        ]
    },
    {
        "question": "How do the original RAG paper and Self-RAG differ in when retrieval is performed?",
        "expected_paper_ids": [
            "rag_lewis_2020",
            "self_rag_asai_2023"
        ]
    },
    {
        "question": "Compare the retrieval ideas in the RAG survey with those proposed by CRAG.",
        "expected_paper_ids": [
            "rag_survey_gao_2023",
            "crag_yan_2024"
        ]
    },
    {
        "question": "How are embeddings and reranking used across DPR and the Qwen3 Embedding paper?",
        "expected_paper_ids": [
            "dpr_karpukhin_2020",
            "qwen3_embedding_2025"
        ]
    }
]

In [ ]:
def evaluate_multi_paper_recall(
    evaluation_questions,
    embedding_model,
    client,
    collection_name,
    bm25,
    chunks,
    reranker,
    k=5
):
    recalls = []
    rows = []

    for item in evaluation_questions:
        query = item["question"]
        expected_papers = set(item["expected_paper_ids"])

        results = smart_retrieve(
            query=query,
            embedding_model=embedding_model,
            client=client,
            collection_name=collection_name,
            bm25=bm25,
            chunks=chunks,
            reranker=reranker,
            top_k=k
        )

        retrieved_papers = [
            result["chunk"]["paper_id"]
            for result in results
        ]

        retrieved_unique = set(retrieved_papers)

        matched = expected_papers.intersection(retrieved_unique)

        recall = len(matched) / len(expected_papers)
        recalls.append(recall)

        rows.append({
            "question": query,
            "expected_papers": list(expected_papers),
            "retrieved_papers": retrieved_papers,
            "matched_papers": list(matched),
            "recall": recall
        })

    mean_recall = sum(recalls) / len(recalls)

    return mean_recall, rows

In [ ]:
multi_recall_5, multi_rows = evaluate_multi_paper_recall(
    evaluation_questions=multi_paper_questions,
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    k=5
)

print("Multi-paper Recall@5:", multi_recall_5)

In [ ]:
for row in multi_rows:
    if row["recall"] < 1.0:
        print("QUESTION:", row["question"])
        print("EXPECTED:", row["expected_papers"])
        print("RETRIEVED:", row["retrieved_papers"])
        print("MATCHED:", row["matched_papers"])
        print("RECALL:", row["recall"])
        print("-" * 100)

In [ ]:
import json

RESULTS_PATH = PROJECT_ROOT / "data" / "evaluation" / "retrieval_metrics.json"

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    evaluation_results = json.load(f)

evaluation_results["multi_paper_recall_at_5"] = multi_recall_5

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_results,
        f,
        indent=2
    )

print("Saved multi-paper Recall@5:", multi_recall_5)